In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# To keep track on model with best columns/score ratio
model_track = {}

# Importing dataset

In [27]:
df_train = pd.read_csv('Marathon_dataset/train.csv')

In [28]:
df_train.head()

,runner_id,age,gender,running_experience_months,previous_marathon_count,training_program,motivation_level,personal_best_minutes,weekly_mileage_km,weekly_mileage_miles,...,run_club_attendance_rate,warmup_adherence_pct,stretching_adherence_pct,marathon_date,marathon_weather,course_difficulty,target_finish_time_minutes,actual_finish_time_minutes,mental_preparation_score,medal_outcome
0,R009432,25,Male,48,0,Intermediate,4,NaN,20.0,12.4,...,0,70,29,2025-02-02,Hot,Mixed,254,297.0,5,0
1,R077741,43,Female,151,2,Advanced,1,196.0,20.0,12.4,...,0,94,0,2024-02-11,Sunny,Mixed,201,231.0,7,1
2,R034376,32,Female,82,3,Advanced,6,211.0,23.1,14.4,...,54,47,39,2025-03-27,Rainy,Flat,204,234.0,9,1
3,R020754,57,Male,6,0,Beginner,8,NaN,27.3,17.0,...,63,77,45,2024-04-16,Sunny,Hilly,258,276.0,3,3
4,R001301,54,Female,11,0,Beginner,3,NaN,20.0,12.4,...,20,64,40,2025-10-21,Cloudy,Mixed,245,296.0,8,0


# Preprocessing

In [29]:
df_prep = df_train.copy()
df_prep.head()

,runner_id,age,gender,running_experience_months,previous_marathon_count,training_program,motivation_level,personal_best_minutes,weekly_mileage_km,weekly_mileage_miles,...,run_club_attendance_rate,warmup_adherence_pct,stretching_adherence_pct,marathon_date,marathon_weather,course_difficulty,target_finish_time_minutes,actual_finish_time_minutes,mental_preparation_score,medal_outcome
0,R009432,25,Male,48,0,Intermediate,4,NaN,20.0,12.4,...,0,70,29,2025-02-02,Hot,Mixed,254,297.0,5,0
1,R077741,43,Female,151,2,Advanced,1,196.0,20.0,12.4,...,0,94,0,2024-02-11,Sunny,Mixed,201,231.0,7,1
2,R034376,32,Female,82,3,Advanced,6,211.0,23.1,14.4,...,54,47,39,2025-03-27,Rainy,Flat,204,234.0,9,1
3,R020754,57,Male,6,0,Beginner,8,NaN,27.3,17.0,...,63,77,45,2024-04-16,Sunny,Hilly,258,276.0,3,3
4,R001301,54,Female,11,0,Beginner,3,NaN,20.0,12.4,...,20,64,40,2025-10-21,Cloudy,Mixed,245,296.0,8,0


In [ ]:
df_prep = df_prep.drop_duplicates()
df_prep = df_prep.dropna(subset=['actual_finish_time_minutes'])


cols_median = ['personal_best_minutes', 'vo2_max', 'cross_training_hours_per_week']
for col in cols_median:
    df_prep[col] = df_prep[col].fillna(df_prep[col].median())

cols_avg = ['sleep_hours_avg', 'nutrition_score', 'hydration_consistency']
for col in cols_avg:
    df_prep[col] = df_prep[col].fillna(df_prep[col].mean())

df_prep['injury_severity'] = df_prep['injury_severity'].fillna(0)

In [31]:
(df_prep.isna().sum() / len(df_prep) * 100)[df_prep.isna().sum() > 0].round(2)

Series([], dtype: float64)

In [32]:
# 1. Ordinal mappings
training_map = {"Beginner": 1, "Intermediate": 2, "Advanced": 3}
course_map = {"Flat": 1, "Mixed": 2, "Hilly": 3}
injury_map = {"Minor": 1, "Moderate": 2, "Severe": 3}

df_prep['training_program'] = df_prep['training_program'].map(training_map)
df_prep['course_difficulty'] = df_prep['course_difficulty'].map(course_map)
df_prep['injury_severity'] = df_prep['injury_severity'].map(injury_map).fillna(0)

# 2. Nominal — one-hot on full df
ohe = OneHotEncoder(drop='first', sparse_output=False)
ohe_array = ohe.fit_transform(df_prep[['gender', 'marathon_weather']])
ohe_df = pd.DataFrame(ohe_array,
                      columns=ohe.get_feature_names_out(['gender', 'marathon_weather']),
                      index=df_prep.index)

df_prep = df_prep.drop(columns=['gender', 'marathon_weather'])
df_prep = pd.concat([df_prep, ohe_df], axis=1)

In [33]:
df_prep.head()

,runner_id,age,running_experience_months,previous_marathon_count,training_program,motivation_level,personal_best_minutes,weekly_mileage_km,weekly_mileage_miles,runs_per_week,...,actual_finish_time_minutes,mental_preparation_score,medal_outcome,gender_Male,gender_Non-binary,marathon_weather_Cold,marathon_weather_Hot,marathon_weather_Rainy,marathon_weather_Sunny,marathon_weather_Windy
0,R009432,25,48,0,2,4,261.0,20.0,12.4,5,...,297.0,5,0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,R077741,43,151,2,3,1,196.0,20.0,12.4,5,...,231.0,7,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,R034376,32,82,3,3,6,211.0,23.1,14.4,6,...,234.0,9,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,R020754,57,6,0,1,8,261.0,27.3,17.0,5,...,276.0,3,3,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,R001301,54,11,0,1,3,261.0,20.0,12.4,5,...,296.0,8,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [109]:
abs(df.corr()['actual_finish_time_minutes'].drop('actual_finish_time_minutes')).sort_values().head(25)

sleep_hours_avg                  0.014306
nutrition_score                  0.027190
early_morning_run_frequency      0.035269
consecutive_weeks_no_miss        0.039239
age                              0.042137
run_club_attendance_rate         0.046154
missed_workout_pct               0.056975
cross_training_hours_per_week    0.090323
course_difficulty                0.100117
weekly_mileage_km                0.112104
long_run_distance_km             0.138232
injury_count                     0.150054
injury_severity                  0.193457
resting_heart_rate_bpm           0.198988
vo2_max                          0.210262
rest_days_per_week               0.298383
runs_per_week                    0.311368
speed_work_sessions_per_week     0.350391
running_experience_months        0.649845
training_program                 0.748738
Name: actual_finish_time_minutes, dtype: float64

In [108]:
df.shape

(78011, 21)

# Dropping Columns 

In [92]:
df_clean = df_prep.copy()
df_prep = df_clean.copy()

In [93]:
minimum_drop_list = ["runner_id",
                     "weekly_mileage_miles",
                     "target_finish_time_minutes",
                     "medal_outcome",
                     "marathon_date",
                     "personal_best_minutes"]

In [94]:
df = df_prep.drop(columns=minimum_drop_list)

## testing different columns drop

In [ ]:
additional_drop = ['gender_Male',
                   'hydration_consistency',
                   'gender_Non-binary',
                   'bmi',
                   'previous_marathon_count',
                   'training_streak_days',
                   'training_adherence_pct',
                   'stretching_adherence_pct',
                   'mental_preparation_score',
                   'motivation_level',
                   'goal_completion_rate',
                   'warmup_adherence_pct',
                   'marathon_weather_Cold',
                   'marathon_weather_Rainy',
                   'marathon_weather_Windy',
                   'marathon_weather_Sunny',
                   'marathon_weather_Hot',
                   'weather_condition_training_pct',
                   'recovery_score']

In [96]:
df = df.drop(columns=additional_drop)

In [97]:
df.columns

Index(['age', 'running_experience_months', 'training_program',
       'weekly_mileage_km', 'runs_per_week', 'long_run_distance_km',
       'speed_work_sessions_per_week', 'rest_days_per_week',
       'consecutive_weeks_no_miss', 'cross_training_hours_per_week',
       'resting_heart_rate_bpm', 'vo2_max', 'sleep_hours_avg', 'injury_count',
       'injury_severity', 'nutrition_score', 'missed_workout_pct',
       'early_morning_run_frequency', 'run_club_attendance_rate',
       'course_difficulty', 'actual_finish_time_minutes'],
      dtype='object')

# Separating X and y

In [98]:
X = df.drop(columns=['actual_finish_time_minutes'])
y = df['actual_finish_time_minutes']

In [99]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Creating a Base Model

In [100]:
ss_scaler = StandardScaler()

In [101]:
X_train_scaled = ss_scaler.fit_transform(X_train)
X_test_scaled = ss_scaler.transform(X_test)

In [102]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [103]:
y_pred = model.predict(X_test_scaled)
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.2f} minutes")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} minutes")
print(f"R²:   {r2_score(y_test, y_pred):.4f}")

MAE:  16.37 minutes
RMSE: 20.48 minutes
R²:   0.6252


In [104]:
model_track[f"test_{len(model_track)+1}"] = {
    "additional_drop": additional_drop,
    "MAE": round(mean_absolute_error(y_test, y_pred), 2),
    "R²": round(r2_score(y_test, y_pred), 4)
}

In [105]:
model_track

{'test_1': {'additional_drop': ['gender_Male',
   'gender_Non-binary',
   'hydration_consistency',
   'previous_marathon_count',
   'training_streak_days',
   'bmi',
   'training_adherence_pct',
   'stretching_adherence_pct',
   'mental_preparation_score',
   'motivation_level',
   'goal_completion_rate',
   'warmup_adherence_pct'],
  'MAE': 16.15,
  'R²': 0.6342},
 'test_2': {'additional_drop': ['gender_Male',
   'gender_Non-binary',
   'hydration_consistency',
   'previous_marathon_count',
   'training_streak_days',
   'bmi',
   'training_adherence_pct',
   'stretching_adherence_pct',
   'mental_preparation_score',
   'motivation_level',
   'goal_completion_rate',
   'warmup_adherence_pct'],
  'MAE': 16.15,
  'R²': 0.6342},
 'test_3': {'additional_drop': ['gender_Male',
   'gender_Non-binary',
   'hydration_consistency',
   'previous_marathon_count',
   'training_streak_days',
   'bmi',
   'training_adherence_pct',
   'stretching_adherence_pct',
   'mental_preparation_score',
   'mot

### To keep track:
with minimal_drop:
MAE:  16.34 minutes
RMSE: 20.39 minutes
R²:   0.6210

# Testing it

In [110]:
df_test = pd.read_csv('Marathon_dataset/test.csv')

In [111]:
df_test = df_test.drop(columns=minimum_drop_list)
df_test = df_test.drop(columns=additional_drop)

KeyError: "['medal_outcome'] not found in axis"

In [ ]:
#X_real_test = df.drop(columns=['actual_finish_time_minutes'])
#y_real_test = df['actual_finish_time_minutes']

In [ ]:
#X_real_test_scaled = ss_scaler.transform(X_real_test)

#y_real_pred = model.predict(X_real_test_scaled)

#print(f"MAE:  {mean_absolute_error(y_real_test, y_real_pred):.2f} minutes")
#print(f"RMSE: {np.sqrt(mean_squared_error(y_real_test, y_real_pred)):.2f} minutes")
#print(f"R²:   {r2_score(y_real_test, y_real_pred):.4f}")